# Stage 6: ROAD Dataset Cleaning and Preparation

Parses the ROAD (Oak Ridge) CAN bus dataset into the same shape as the
CICIoV2024 pipeline, then runs the identical de-duplication, split, and
preprocessing used for CICIoV.

**Why ROAD:** a second, independent CAN dataset lets us test whether the
de-duplication findings and adversarial behaviour generalise beyond CICIoV2024.

**Trainable classes (5):** benign, max-speedometer, reverse-light-on,
reverse-light-off, fuzzing.

**Excluded:** correlated-signal collapses to a single unique signature under
strict de-duplication (fully-specified fixed payload) and is not statistically
viable to train or test. It is retained only as a reported de-duplication
statistic. Masquerade variants are excluded: they duplicate fabrication
signatures rather than adding diversity.

**Labelling:** targeted attacks labelled by injection ID + time interval + fixed
payload bytes; fuzzing labelled by interval + all-FF payload filter to isolate
true injections from coincident real traffic.

In [1]:
# This notebook lives in notebooks/; code lives in src/
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

# Project modules
import config
import road_cleaning
import cleaning
import preprocessing

# Other imports
import pandas as pd
import numpy as np
import json
import joblib

print("Imports OK. Project root:", config.PROJECT_ROOT)

Imports OK. Project root: /home/koala/lab/adversec


## Parse and build the ROAD dataset

Parses all attack captures (targeted + fuzzing), pools them by clean class name,
samples benign frames from two ambient captures (dyno + real-road), and returns
one DataFrame in CICIoV shape: ID, DATA_0..DATA_7, true_class.

In [2]:
# Build the full ROAD dataset (attacks + benign) in CICIoV shape
road_df = road_cleaning.load_road_dataset(
    config.ROAD_ATTACK_CAPTURES,
    config.ROAD_AMBIENT_CAPTURES,
    config.ROAD_ATTACKS_DIR,
    config.ROAD_AMBIENT_DIR,
    config.ROAD_ATTACKS_DIR / "capture_metadata.json",
    config.FEATURE_COLUMNS,
    config.ROAD_AMBIENT_SAMPLE_PER_CAPTURE,
    config.RANDOM_SEED,
)

print("\ncolumns:", list(road_df.columns))

max-speedometer      raw= 11689  pooled_unique_signatures=10559
reverse-light-on     raw=  8032  pooled_unique_signatures=5994
reverse-light-off    raw=  5476  pooled_unique_signatures=1525
fuzzing              raw=  1055  pooled_unique_signatures=592
ambient_dyno_drive_basic_long            sampled= 20000
ambient_highway_street_driving_long      sampled= 20000

full ROAD dataset shape: (66252, 10)
true_class
benign               40000
max-speedometer      11689
reverse-light-on      8032
reverse-light-off     5476
fuzzing               1055
Name: count, dtype: int64

columns: ['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3', 'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7', 'true_class']


## Strict de-duplication and the diversity gradient

Applies the same signature-level de-duplication used for CICIoV2024. The
per-class unique-signature counts form the scarcity gradient that later analysis
uses. The excluded correlated-signal capture is measured here too, to record its
collapse to a single signature as the documented floor of the gradient.

In [3]:
# --- Strict de-duplication (same function as CICIoV) ---
road_strict = cleaning.strict_dedup(road_df, config.FEATURE_COLUMNS)

print("before dedup:", road_df.shape)
print("after  dedup:", road_strict.shape)
print("\nunique signatures per trainable class:")
gradient = road_strict["true_class"].value_counts()
print(gradient)

# --- Measure the excluded correlated-signal floor ---
# Parsed separately because it is not in ROAD_ATTACK_CAPTURES (excluded from
# modelling). We record its single-signature collapse as the gradient floor.
meta = road_cleaning.load_metadata(config.ROAD_ATTACKS_DIR / "capture_metadata.json")
corr_captures = ["correlated_signal_attack_1",
                 "correlated_signal_attack_2",
                 "correlated_signal_attack_3"]
corr_pieces = []
for cap in corr_captures:
    d = road_cleaning.parse_log_file(config.ROAD_ATTACKS_DIR / f"{cap}.log")
    d = road_cleaning.label_attack_capture(d, cap, meta)
    corr_pieces.append(d[d["true_class"] == cap].copy())
corr = pd.concat(corr_pieces, ignore_index=True)
corr_raw = len(corr)
corr_sigs = len(corr[config.FEATURE_COLUMNS].drop_duplicates())
print(f"\nEXCLUDED correlated-signal: raw={corr_raw}  unique_signatures={corr_sigs}")

# --- Save the gradient statistic to results/ ---
gradient_report = {
    "dataset": "ROAD",
    "dedup": "strict signature-level (features + true_class)",
    "total_frames_before_dedup": int(len(road_df)),
    "total_signatures_after_dedup": int(len(road_strict)),
    "trainable_classes": {k: int(v) for k, v in gradient.items()},
    "excluded": {
        "correlated-signal": {
            "raw_frames": int(corr_raw),
            "unique_signatures": int(corr_sigs),
            "reason": "fully-specified fixed payload collapses to a single signature; not viable to train or test",
        },
        "masquerade_variants": {
            "reason": "built by deleting target-ID legitimate frames; injected frames duplicate fabrication signatures rather than adding diversity",
        },
    },
}

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = config.RESULTS_DIR / "road_dedup_gradient.json"
with open(report_path, "w") as f:
    json.dump(gradient_report, f, indent=2)
print(f"\nsaved gradient report -> {report_path}")

before dedup: (66252, 10)
after  dedup: (39858, 10)

unique signatures per trainable class:
true_class
benign               21188
max-speedometer      10559
reverse-light-on      5994
reverse-light-off     1525
fuzzing                592
Name: count, dtype: int64

EXCLUDED correlated-signal: raw=5490  unique_signatures=1

saved gradient report -> /home/koala/lab/adversec/results/road_dedup_gradient.json


## Split, preprocess, and save processed artefacts

Signature-level train/test split (no signature crosses the boundary), light
duplication on train only (barely fires here, all classes exceed the target),
then label encoding and [0,1] feature scaling fitted on train only. Saves the
processed arrays and fitted objects with a road_ prefix, mirroring the CICIoV2024
Stage 2 outputs, so the baseline and adversarial notebooks can load them directly.

In [4]:
# --- Signature-level split (same function as CICIoV) ---
road_train, road_test = cleaning.split_train_test(
    road_strict, config.FEATURE_COLUMNS,
    test_fraction=0.2, random_seed=config.RANDOM_SEED,
)
print("train signatures:", road_train.shape)
print("test  signatures:", road_test.shape)

# --- Light duplication on train only (barely fires; classes already large) ---
road_train_dup = cleaning.duplicate_train_classes(
    road_train, target_per_class=200, random_seed=config.RANDOM_SEED,
)
print("train after duplication:", road_train_dup.shape)

# --- Leakage check: no signature shared between train and test ---
train_sigs = set(map(tuple, road_train[config.FEATURE_COLUMNS].itertuples(index=False, name=None)))
test_sigs  = set(map(tuple, road_test[config.FEATURE_COLUMNS].itertuples(index=False, name=None)))
print("overlapping signatures:", len(train_sigs & test_sigs))

# --- Encode labels and scale features (fit on train only) ---
y_train, y_test, road_encoder = preprocessing.encode_labels(road_train_dup, road_test)
X_train, X_test, road_scaler = preprocessing.scale_features(
    road_train_dup, road_test, config.FEATURE_COLUMNS,
)
print("\nX_train:", X_train.shape, " X_test:", X_test.shape)

# --- Save processed artefacts (road_ prefix, mirrors CICIoV Stage 2) ---
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

road_strict.to_csv(config.PROCESSED_DIR / "road_strict.csv", index=False)
road_train_dup.to_csv(config.PROCESSED_DIR / "road_train_dup.csv", index=False)
road_test.to_csv(config.PROCESSED_DIR / "road_test.csv", index=False)

np.savez_compressed(
    config.PROCESSED_DIR / "road_stage2_arrays.npz",
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)
joblib.dump(road_encoder, config.PROCESSED_DIR / "road_label_encoder.joblib")
joblib.dump(road_scaler, config.PROCESSED_DIR / "road_feature_scaler.joblib")

print("\nSaved:")
for name in ["road_strict.csv", "road_train_dup.csv", "road_test.csv",
             "road_stage2_arrays.npz", "road_label_encoder.joblib",
             "road_feature_scaler.joblib"]:
    print("  ", config.PROCESSED_DIR / name)

train signatures: (31886, 10)
test  signatures: (7972, 10)
train after duplication: (31886, 10)
overlapping signatures: 0
Label mapping:
    0 -> benign
    1 -> fuzzing
    2 -> max-speedometer
    3 -> reverse-light-off
    4 -> reverse-light-on

X_train: (31886, 9)  X_test: (7972, 9)

Saved:
   /home/koala/lab/adversec/datasets/processed/road_strict.csv
   /home/koala/lab/adversec/datasets/processed/road_train_dup.csv
   /home/koala/lab/adversec/datasets/processed/road_test.csv
   /home/koala/lab/adversec/datasets/processed/road_stage2_arrays.npz
   /home/koala/lab/adversec/datasets/processed/road_label_encoder.joblib
   /home/koala/lab/adversec/datasets/processed/road_feature_scaler.joblib
